In [6]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
# Load Original Data
def load_and_clean_project_data():
    # Load in the census data
    census_data = pd.read_csv("data/census_data.csv", skiprows=1)
    census_data

    # Load in the Internet data
    internet_data = pd.read_csv("data/internet_data.tsv", sep="\t")
    internet_data.head()

    # Load in the ecommerce data
    ecommerce_data = pd.read_csv("data/ecommerce_data.csv")
    ecommerce_data.head()

    census_data_filtered = census_data[['Geographic Area Name','Estimate!!Households!!Median income (dollars)']]

    # Clean the Zip Code Column so it can be matched
    census_data_filtered['Geographic Area Name'] = (
        census_data_filtered['Geographic Area Name']
            .astype(str)
            .str[-5:]
    )
    census_data_filtered

    # Rename Column Names to be More Representative and use _ for analysis
    census_data_filtered = census_data_filtered.rename(
        columns={'Geographic Area Name': 'Zip_codes'}
    )
    census_data_filtered

    # Rename Column Names to be More Representative and use _ for analysis
    census_data_filtered = census_data_filtered.rename(
        columns={'Estimate!!Households!!Median income (dollars)': 'Median_household_income'}
    )
    census_data_filtered

    # There are some empty values in this column that are represented by - or **, let's take care of those
    bad_vals = ["-", "**"]
    census_data_cleaned = census_data_filtered[~census_data_filtered['Median_household_income'].isin(bad_vals)]
    census_data_cleaned['Median_household_income'] = census_data_cleaned['Median_household_income'].replace('250,000+', '250000')
    census_data_cleaned['Median_household_income'] = census_data_cleaned['Median_household_income'].replace('2,500-', '2500')

    # Change Median Household Income column into an int
    census_data_cleaned['Median_household_income'] = census_data_cleaned['Median_household_income'].astype(int)

    # The zip code column should be of type str and not type int
    internet_data['ZCTA19'] = internet_data['ZCTA19'].astype(str)

    # The zip code column should be of type str and not type int
    len_list = []
    for row in internet_data['ZCTA19']:
        if len(row) not in len_list:
            len_list.append(len(row))
    internet_data['ZCTA19'] = internet_data['ZCTA19'].astype(str).str.zfill(5)

    # The zip code column should be of type str and not type int
    len_list = []
    for row in internet_data['ZCTA19']:
        if len(row) not in len_list:
            len_list.append(len(row))

    # Rename Column Names to be More Representative and use _ for analysis
    internet_data_cleaned = internet_data.rename(
        columns={'ZCTA19': 'Zip_codes'}
    )

    # Rename Column Names to be More Representative and use _ for analysis
    internet_data_cleaned = internet_data_cleaned.rename(
        columns={'HOUSEHOLDS': 'Total_households'}
    )
    # Rename Column Names to be More Representative and use _ for analysis
    internet_data_cleaned = internet_data_cleaned.rename(
        columns={'N_INTERNET_SUB': 'Households_w_internet_access'}
    )

    # Only selectint the relevant columns from the internet data
    internet_data_filtered = internet_data_cleaned[['Zip_codes', 'Total_households', 'Households_w_internet_access']]

    # Add a column to Show Percentage of Households with Internet Access
    internet_data_filtered['Internet_access'] = internet_data_filtered['Households_w_internet_access'] / internet_data_filtered['Total_households']

    # Drop the null values
    internet_data_filtered = internet_data_filtered.dropna()

    # Switching the Postal Code to be a string data type
    ecommerce_data['Postal Code'] = ecommerce_data['Postal Code'].astype('string')
    # Ensure 5-character postal codes with leading zeros
    ecommerce_data["Postal Code"] = ecommerce_data["Postal Code"].astype(str).str.zfill(5)
    # Group ecommerce data by postal code
    ecommerce_data_filtered = ecommerce_data.groupby('Postal Code').size().reset_index(name="order_count")

    # Rename Column Names to be More Representative and use _ for analysis
    ecommerce_data_filtered = ecommerce_data_filtered.rename(
        columns={'Postal Code': 'Zip_codes'}
    )
    ecommerce_data_filtered

    merged_df = (
        census_data_filtered
        .merge(internet_data_filtered, on='Zip_codes', how='inner')
        .merge(ecommerce_data_filtered, on='Zip_codes', how='inner')
    )

    # Change Median Household Income column into an int
    merged_df['Median_household_income'] = merged_df['Median_household_income'].astype(int)
    return merged_df
    

In [8]:
merged_df = load_and_clean_project_data()

/tmp/ipykernel_80841/3895609181.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  census_data_filtered['Geographic Area Name'] = (
/tmp/ipykernel_80841/3895609181.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  census_data_cleaned['Median_household_income'] = census_data_cleaned['Median_household_income'].replace('250,000+', '250000')
/tmp/ipykernel_80841/3895609181.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

In [9]:
# Load in the zip data
zip_data = pd.read_csv("data/uszips.csv")
zip_data

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,county_fips,county_name,county_weights,county_names_all,county_fips_all,imprecise,military,timezone
0,601,18.18027,-66.75266,Adjuntas,PR,Puerto Rico,True,NaN,16669.0,99.9,72001,Adjuntas,"{""72001"": 98.74, ""72141"": 1.26}",Adjuntas|Utuado,72001|72141,False,False,America/Puerto_Rico
1,602,18.36075,-67.17541,Aguada,PR,Puerto Rico,True,NaN,37233.0,474.0,72003,Aguada,"{""72003"": 100}",Aguada,72003,False,False,America/Puerto_Rico
2,603,18.45744,-67.12225,Aguadilla,PR,Puerto Rico,True,NaN,48448.0,544.6,72005,Aguadilla,"{""72005"": 99.76, ""72099"": 0.24}",Aguadilla|Moca,72005|72099,False,False,America/Puerto_Rico
3,606,18.16585,-66.93716,Maricao,PR,Puerto Rico,True,NaN,5163.0,45.0,72093,Maricao,"{""72093"": 82.26, ""72153"": 11.67, ""72121"": 6.06}",Maricao|Yauco|Sabana Grande,72093|72153|72121,False,False,America/Puerto_Rico
4,610,18.29110,-67.12243,Anasco,PR,Puerto Rico,True,NaN,25357.0,263.8,72011,Añasco,"{""72011"": 96.8, ""72099"": 2.83, ""72083"": 0.37}",Añasco|Moca|Las Marías,72011|72099|72083,False,False,America/Puerto_Rico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33777,99774,66.02320,-149.07982,Stevens Village,AK,Alaska,True,NaN,0.0,0.0,2290,Yukon-Koyukuk,"{""02290"": 100}",Yukon-Koyukuk,02290,False,False,America/Anchorage
33778,99778,65.25309,-166.34200,Teller,AK,Alaska,True,NaN,255.0,66.0,2180,Nome,"{""02180"": 100}",Nome,02180,False,False,America/Nome
33779,99790,65.42642,-148.18738,Fairbanks,AK,Alaska,True,NaN,0.0,0.0,2290,Yukon-Koyukuk,"{""02290"": 100}",Yukon-Koyukuk,02290,False,False,America/Anchorage
33780,99850,58.48821,-135.48432,Juneau,AK,Alaska,True,NaN,0.0,0.0,2100,Haines,"{""02100"": 100}",Haines,02100,False,False,America/Juneau


In [10]:
zip_data_clean = zip_data.copy()

In [11]:
# Clean the Zip Code Column so it can be matched
zip_data_clean["zip"] = (
    zip_data_clean["zip"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
)
zip_data_clean

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,county_fips,county_name,county_weights,county_names_all,county_fips_all,imprecise,military,timezone
0,00601,18.18027,-66.75266,Adjuntas,PR,Puerto Rico,True,NaN,16669.0,99.9,72001,Adjuntas,"{""72001"": 98.74, ""72141"": 1.26}",Adjuntas|Utuado,72001|72141,False,False,America/Puerto_Rico
1,00602,18.36075,-67.17541,Aguada,PR,Puerto Rico,True,NaN,37233.0,474.0,72003,Aguada,"{""72003"": 100}",Aguada,72003,False,False,America/Puerto_Rico
2,00603,18.45744,-67.12225,Aguadilla,PR,Puerto Rico,True,NaN,48448.0,544.6,72005,Aguadilla,"{""72005"": 99.76, ""72099"": 0.24}",Aguadilla|Moca,72005|72099,False,False,America/Puerto_Rico
3,00606,18.16585,-66.93716,Maricao,PR,Puerto Rico,True,NaN,5163.0,45.0,72093,Maricao,"{""72093"": 82.26, ""72153"": 11.67, ""72121"": 6.06}",Maricao|Yauco|Sabana Grande,72093|72153|72121,False,False,America/Puerto_Rico
4,00610,18.29110,-67.12243,Anasco,PR,Puerto Rico,True,NaN,25357.0,263.8,72011,Añasco,"{""72011"": 96.8, ""72099"": 2.83, ""72083"": 0.37}",Añasco|Moca|Las Marías,72011|72099|72083,False,False,America/Puerto_Rico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33777,99774,66.02320,-149.07982,Stevens Village,AK,Alaska,True,NaN,0.0,0.0,2290,Yukon-Koyukuk,"{""02290"": 100}",Yukon-Koyukuk,02290,False,False,America/Anchorage
33778,99778,65.25309,-166.34200,Teller,AK,Alaska,True,NaN,255.0,66.0,2180,Nome,"{""02180"": 100}",Nome,02180,False,False,America/Nome
33779,99790,65.42642,-148.18738,Fairbanks,AK,Alaska,True,NaN,0.0,0.0,2290,Yukon-Koyukuk,"{""02290"": 100}",Yukon-Koyukuk,02290,False,False,America/Anchorage
33780,99850,58.48821,-135.48432,Juneau,AK,Alaska,True,NaN,0.0,0.0,2100,Haines,"{""02100"": 100}",Haines,02100,False,False,America/Juneau


In [12]:
merged_df_clean = merged_df.copy()

In [13]:
merged_df_with_state = merged_df_clean.merge(
    zip_data_clean[["zip", "state_id", "state_name"]],
    left_on="Zip_codes",
    right_on="zip",
    how="left"
)
merged_df_with_state

,Zip_codes,Median_household_income,Total_households,Households_w_internet_access,Internet_access,order_count,zip,state_id,state_name
0,01841,45486,15149,10693,0.705855,12,01841,MA,Massachusetts
1,01852,65445,14255,10844,0.760716,7,01852,MA,Massachusetts
2,02038,118193,11941,11205,0.938364,2,02038,MA,Massachusetts
3,02138,104641,14188,12513,0.881942,2,02138,MA,Massachusetts
4,02149,70627,16021,13483,0.841583,9,02149,MA,Massachusetts
...,...,...,...,...,...,...,...,...,...
432,98502,74031,14290,12469,0.872568,2,98502,WA,Washington
433,98632,51188,20963,17051,0.813385,2,98632,WA,Washington
434,98661,59466,18797,16371,0.870937,1,98661,WA,Washington
435,99207,42913,12204,10541,0.863733,4,99207,WA,Washington


In [14]:
state_df = (
    merged_df_with_state
    .groupby("state_name", as_index=False)
    .agg(
        order_count=("order_count", "sum"),
        total_households=("Total_households", "sum"),
        households_w_internet=("Households_w_internet_access", "sum"),
        median_household_income=(
            "Median_household_income",
            lambda x: (
                (x * merged_df_with_state.loc[x.index, "Total_households"]).sum()
                / merged_df_with_state.loc[x.index, "Total_households"].sum()
            )
        )
    )
)
state_df.head(5)

,state_name,order_count,total_households,households_w_internet,median_household_income
0,Alabama,16,94851,74721,49597.996690
1,Arizona,70,202650,166065,56085.661826
2,Arkansas,19,108072,80689,46864.679658
3,California,663,1174613,1025516,84823.705596
4,Colorado,61,182497,162756,81025.059256


In [15]:
state_df['internet_access'] = state_df['households_w_internet']/state_df['total_households']
state_df.head(5)

,state_name,order_count,total_households,households_w_internet,median_household_income,internet_access
0,Alabama,16,94851,74721,49597.996690,0.787772
1,Arizona,70,202650,166065,56085.661826,0.819467
2,Arkansas,19,108072,80689,46864.679658,0.746623
3,California,663,1174613,1025516,84823.705596,0.873067
4,Colorado,61,182497,162756,81025.059256,0.891828


In [17]:
import pandas as pd

covid_df = pd.read_csv("data/us-states.csv")
covid_df["date"] = pd.to_datetime(covid_df["date"])
covid_2020 = covid_df[covid_df["date"].dt.year == 2020]


In [21]:
covid_state_2020 = (
    covid_2020
    .groupby("state", as_index=False)
    .agg(
        total_cases=("cases", "max"),
        total_deaths=("deaths", "max")
    )
)
covid_state_2020.head(5)

,state,total_cases,total_deaths
0,Alabama,361226,4827
1,Alaska,46740,198
2,Arizona,523829,8879
3,Arkansas,225138,3676
4,California,2307860,25965


In [23]:
merged_with_covid = state_df.merge(
    covid_state_2020,
    left_on="state_name",
    right_on="state",
    how="left"
)
merged_with_covid.head(5)

,state_name,order_count,total_households,households_w_internet,median_household_income,internet_access,state,total_cases,total_deaths
0,Alabama,16,94851,74721,49597.996690,0.787772,Alabama,361226,4827
1,Arizona,70,202650,166065,56085.661826,0.819467,Arizona,523829,8879
2,Arkansas,19,108072,80689,46864.679658,0.746623,Arkansas,225138,3676
3,California,663,1174613,1025516,84823.705596,0.873067,California,2307860,25965
4,Colorado,61,182497,162756,81025.059256,0.891828,Colorado,335579,4879


In [24]:
# Normalize All Columns

merged_with_covid["covid_cases_per_household"] = (
    merged_with_covid["total_cases"] / merged_with_covid["total_households"]
)

merged_with_covid["covid_deaths_per_household"] = (
    merged_with_covid["total_deaths"] / merged_with_covid["total_households"]
)

merged_with_covid["orders_per_household"] = (
    merged_with_covid["order_count"] / merged_with_covid["total_households"]
)


In [41]:
# Features
X_cols = merged_with_covid[
    [
        "median_household_income",
        "internet_access",
        "covid_deaths_per_household",
        "covid_cases_per_household"
    ]
]

# Target: orders per household
target_col = merged_with_covid[["orders_per_household"]]

In [42]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X_cols,
    target_col,
    test_size=0.3,
    random_state=42
)

In [43]:
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 5]
}

In [44]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

rcv = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=0,
        n_jobs=1   # important for Codespaces stability
    ),
    param_distributions=param_grid,
    n_iter=10,
    scoring="r2",
    cv=5,
    n_jobs=1,
    random_state=0
)

In [45]:
rcv.fit(x_train, y_train)

print("Best estimator:", rcv.best_estimator_)
print("Best params:", rcv.best_params_)
print("Best CV R2:", rcv.best_score_)

/workspaces/Module-B-semester-2/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/workspaces/Module-B-semester-2/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/workspaces/Module-B-semester-2/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/workspaces/Module-B-semester-2/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vec

Best estimator: RandomForestRegressor(max_depth=10, min_samples_leaf=5, n_jobs=1,
                      random_state=0)
Best params: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_depth': 10}
Best CV R2: -0.8064414743503123


In [46]:
from sklearn.metrics import r2_score

best_model = rcv.best_estimator_
y_pred = best_model.predict(x_test)

print("Test R2:", r2_score(y_test, y_pred))


Test R2: -0.36651976464561664


In [39]:
import pandas as pd

feature_importance = pd.DataFrame(
    {
        "feature": X_cols.columns,
        "importance": best_model.feature_importances_
    }
).sort_values("importance", ascending=False)

feature_importance

,feature,importance
2,covid_cases_per_household,0.467794
0,median_household_income,0.303560
3,orders_per_household,0.159480
1,internet_access,0.069166
